# Config stuff

In [61]:

import ConnectionConfig as cc
from delta import DeltaTable
from datetime import datetime


In [62]:
cc.setupEnvironment()
spark = cc.startLocalCluster("dimUserIncrementalLoad")
spark.getActiveSession()

In [63]:
run_timestamp =datetime.now() #The job runtime is stored in a variable

### Read existing dimension

In [64]:
#EXTRACT
dt_dimUser = DeltaTable.forPath(spark,".\\spark-warehouse\\dimuser")

dt_dimUser.toDF().createOrReplaceTempView("dimUsers_current")

#DEBUG CODE TO SHOW CONTENT OF DIMENSION
spark.sql("select userid, street, user_SK, md5  from dimUsers_current").show()

+------+--------------------+--------------------+--------------------+
|userid|              street|             user_SK|                 md5|
+------+--------------------+--------------------+--------------------+
|    15|    John Kennedylaan|44d090cc-2c55-49c...|6e3779642d13bf4ab...|
|    16|              Wijk 2|9e5bbdc6-87b5-46c...|81dec6ab4f1b91eef...|
|    17|       Schoggestraat|5d9385c6-dd44-43d...|f4b825db3b59e9970...|
|    18|Emiel Van Hemeldo...|33d13736-f5eb-49f...|319ef7ee26c646ab2...|
|    19|            Fortbaan|f81341fc-1d1c-474...|ccc59a80114fbf09b...|
|    20|  Joseph Wateletlaan|142beb13-d2f5-426...|aa838917117b89e28...|
|    21|    Montevideostraat|6cf8aa79-a353-419...|12079fbd7516de0ef...|
|    22|Joseph Van Hellem...|39491f6a-b080-4ac...|4bb954ba48c3c2cb5...|
|    23|Floris de Cuijper...|6c35793c-5323-48d...|4cf051bec68639f95...|
|    24|Camille Huysmanslaan|c09e253d-d399-42a...|2cdebed9c10b9ab8a...|
|    25|      Korenbloemlaan|ff73d716-7085-48b...|aeff6d2cc6b570

In [65]:
cc.set_connectionProfile("velodb")

#EXTRACT
df_operational_sales_rep = spark.read \
    .format("jdbc") \
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "velo_users") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "userid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 20) \
    .load()

df_operational_sales_rep.createOrReplaceTempView("operational_users")

#TRANSFORMt
df_operational_users_new = spark.sql( "select uuid() as user_SK, \
                                        userid as source_userId, \
                                        street as source_street, \
                                      number as source_number, \
                                      zipcode as source_zipcode, \
                                      city as source_city, \
                                      country_code as source_country_code, \
                                        md5(concat(street, number, zipcode, city, country_code)) as source_md5 \
                                    from operational_users")

df_operational_users_new.createOrReplaceTempView("dimUsers_new")

#DEBUG CODE TO SHOW CONTENT OF SOURCE
#df_dim_sales_rep_new.printSchema()
#df_dim_sales_rep_new.show()
spark.sql("select * from dimUsers_new").show()
#df_dim_sales_rep.write.format("delta").mode("overwrite").saveAsTable("dimSalesRep")

from pyspark.sql.functions import when

# Simulate street change for userid 1
df_operational_sales_rep = df_operational_sales_rep.withColumn(
    "street",
    when(df_operational_sales_rep.userid == 1, "New Street Name").otherwise(df_operational_sales_rep.street)
)
df_operational_sales_rep.createOrReplaceTempView("operational_users")



+--------------------+-------------+--------------------+-------------+--------------+--------------------+-------------------+--------------------+
|             user_SK|source_userId|       source_street|source_number|source_zipcode|         source_city|source_country_code|          source_md5|
+--------------------+-------------+--------------------+-------------+--------------+--------------------+-------------------+--------------------+
|f6e3896c-025b-427...|            2|          Europalaan|          43 |          2610| Wilrijk (Antwerpen)|                 BE|e91f35aa29e5d732f...|
|e2717830-ff25-4d6...|            3|   Maria Clarastraat|          80 |          2160|           Wommelgem|                 BE|5372c35ac3d3a8b04...|
|61139d88-46d7-46c...|            4|Graaf Joseph de P...|          15 |          2900|             Schoten|                 BE|92680e7e5a3c54a58...|
|4dd12441-7a79-413...|            1|     New Street Name|         156 |          2060|           Antwerpen

In [66]:
#TRANSFORM
detectedChanges = spark.sql(f"""
SELECT
    source.source_userId,
    dwh.userid,
    source.source_street, dwh.street,
    source.source_number, dwh.number,
    source.source_zipcode, dwh.zipcode,
    source.source_city, dwh.city,
    source.source_country_code, dwh.country_code,
    source.source_md5, dwh.md5,
    dwh.current,
    dwh.user_SK,
    dwh.scd_start,
    dwh.number as dwh_number,
    dwh.zipcode as dwh_zipcode,
    dwh.city as dwh_city,
    dwh.country_code as dwh_country_code
FROM dimUsers_new source
LEFT OUTER JOIN dimUsers_current dwh
    ON dwh.userid = source.source_userId AND dwh.current = true
WHERE dwh.userid IS NOT NULL AND source.source_md5 <> dwh.md5
""")





detectedChanges.createOrReplaceTempView("detectedChanges")

#DEBUG CODE TO SHOW CONTENT OF DETECTED CHANGES
detectedChanges.show()


+-------------+------+---------------+-----------+-------------+------+--------------+-------+-----------+---------+-------------------+------------+--------------------+--------------------+-------+--------------------+-------------------+----------+-----------+---------+----------------+
|source_userId|userid|  source_street|     street|source_number|number|source_zipcode|zipcode|source_city|     city|source_country_code|country_code|          source_md5|                 md5|current|             user_SK|          scd_start|dwh_number|dwh_zipcode| dwh_city|dwh_country_code|
+-------------+------+---------------+-----------+-------------+------+--------------+-------+-----------+---------+-------------------+------------+--------------------+--------------------+-------+--------------------+-------------------+----------+-----------+---------+----------------+
|            1|     1|New Street Name|Somméstraat|         156 |  156 |          2060|   2060|  Antwerpen|Antwerpen|           

In [67]:
#TRANSFORM
spark.sql("""
SELECT

    source.source_userId,
    dwh.userid,
    source.source_md5,
    dwh.md5,
    concat(source.source_street, source.source_number, source.source_zipcode, source.source_city, source.source_country_code) as src_concat,
    concat(dwh.street, dwh.number, dwh.zipcode, dwh.city, dwh.country_code) as dwh_concat
FROM dimUsers_new source
LEFT OUTER JOIN dimUsers_current dwh
    ON dwh.userid = source.source_userId AND dwh.current = true
WHERE dwh.userid IS NOT NULL
""").show(truncate=False)

+-------------+------+--------------------------------+--------------------------------+-------------------------------------------------------------+-------------------------------------------------------------+
|source_userId|userid|source_md5                      |md5                             |src_concat                                                   |dwh_concat                                                   |
+-------------+------+--------------------------------+--------------------------------+-------------------------------------------------------------+-------------------------------------------------------------+
|2            |2     |e91f35aa29e5d732ffff6e597139cea1|e91f35aa29e5d732ffff6e597139cea1|Europalaan43 2610Wilrijk (Antwerpen)BE                       |Europalaan43 2610Wilrijk (Antwerpen)BE                       |
|3            |3     |5372c35ac3d3a8b04fe16d5e4b46dff8|5372c35ac3d3a8b04fe16d5e4b46dff8|Maria Clarastraat80 2160WommelgemBE                         


##### 3 TRANSOFRM TO UPSERTS
Before union: Every updated and new source-row requires the insertion of a new record in the SCD2 dimension. This new records starts at the runtime of the job and ends at the end of time (2100-12-12). Current is set to true.
Updated source-rows also require an update of the existing scd-fields. The scd_end date of the existing record is set to the runtime of the job. Current is set to false

In the next step, rows without mergeKey will be inserted in the dimension table and rows with mergekey will be updated in the dimension

In [68]:
#TRANSFORM
df_upserts = spark.sql(f"""
    SELECT
        uuid() AS user_SK,
        source_userId AS userid,
        source_street AS street,
        source_number,
        source_zipcode AS zipcode,
        source_city AS city,
        source_country_code AS country_code,
        TO_TIMESTAMP('{run_timestamp}') AS scd_start,
        TO_TIMESTAMP('2100-12-12','yyyy-MM-dd') AS scd_end,
        source_md5 AS md5,
        TRUE AS current
    FROM detectedChanges

    UNION

    SELECT
        user_SK,
        userid,
        street,
        number,
        zipcode,
        city,
        country_code,
        scd_start,
        TO_TIMESTAMP('{run_timestamp}') AS scd_end,
        md5,
        FALSE AS current
    FROM detectedChanges
    WHERE current = true
""")




df_upserts.createOrReplaceTempView("upserts")


In [69]:

#DEBUG CODE TO SHOW CONTENT OF UPSERTS
spark.sql("select * from upserts").show()

+--------------------+------+---------------+-------------+-------+---------+------------+--------------------+--------------------+--------------------+-------+
|             user_SK|userid|         street|source_number|zipcode|     city|country_code|           scd_start|             scd_end|                 md5|current|
+--------------------+------+---------------+-------------+-------+---------+------------+--------------------+--------------------+--------------------+-------+
|22e47541-a930-4e9...|     1|New Street Name|         156 |   2060|Antwerpen|          BE|2025-05-08 21:01:...| 2100-12-12 00:00:00|979fb997dc154d62b...|   true|
|e566d819-e9f5-476...|     1|    Somméstraat|         156 |   2060|Antwerpen|          BE| 1999-01-01 00:00:00|2025-05-08 21:01:...|fddd2c34acf178b1f...|  false|
+--------------------+------+---------------+-------------+-------+---------+------------+--------------------+--------------------+--------------------+-------+



In [70]:
#TRANSFORM
spark.sql("""
MERGE INTO dimUsers_current AS target
USING upserts AS source
ON target.userid = source.userid
AND source.current = false
AND target.current = true
WHEN MATCHED THEN
    UPDATE SET
        target.scd_end = source.scd_end,
        target.current = source.current
WHEN NOT MATCHED THEN
    INSERT (
        user_SK, userid, street, number, zipcode, city, country_code,
        scd_start, scd_end, md5, current
    )
    VALUES (
        source.user_SK, source.userid, source.street, source_number,
        source.zipcode, source.city, source.country_code,
        source.scd_start, source.scd_end, source.md5, source.current
    )
""")

#DEBUG CODE TO SHOW CONTENT OF DIMENSION
# dt_dimUser.toDF().sort("userid", "scd_start").show(20)

dt_dimUser.toDF().filter("current = true").show(truncate=False)
dt_dimUser.toDF().filter("current = false").show(truncate=False)




+------+--------------------------+--------+-------+----------------------------------------------------------+------------+------------------------------------+-------------------+-------------------+--------------------------------+-------+
|userid|street                    |number  |zipcode|city                                                      |country_code|user_SK                             |scd_start          |scd_end            |md5                             |current|
+------+--------------------------+--------+-------+----------------------------------------------------------+------------+------------------------------------+-------------------+-------------------+--------------------------------+-------+
|15    |John Kennedylaan          |9 0801  |2520   |Broechem/Emblem/Oelegem/Ranst                             |BE          |44d090cc-2c55-49ca-81ae-2605bf351064|1999-01-01 00:00:00|2100-12-12 00:00:00|6e3779642d13bf4ab0dbbf5996d8c2c9|true   |
|16    |Wijk 2              

In [71]:
dt_dimUser.toDF().filter("userid = 1").orderBy("scd_start").show(truncate=False)


+------+---------------+------+-------+---------+------------+------------------------------------+--------------------------+--------------------------+--------------------------------+-------+
|userid|street         |number|zipcode|city     |country_code|user_SK                             |scd_start                 |scd_end                   |md5                             |current|
+------+---------------+------+-------+---------+------------+------------------------------------+--------------------------+--------------------------+--------------------------------+-------+
|1     |Somméstraat    |156   |2060   |Antwerpen|BE          |e566d819-e9f5-4763-b9ca-b4bc27422ecd|1999-01-01 00:00:00       |2025-05-08 21:01:46.955288|fddd2c34acf178b1f61f33d3e3b5890c|false  |
|1     |New Street Name|156   |2060   |Antwerpen|BE          |22e47541-a930-4e9d-ae10-c8b24a3bce7b|2025-05-08 21:01:46.955288|2100-12-12 00:00:00       |979fb997dc154d62b9b6e6688480df61|true   |
+------+---------------+-

## Delete the spark session

In [34]:
df_upserts.write.format("delta") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("dimUser")

spark.stop()